# M2: remove roots from empty versions

Create a separate derived dataset from a completed M2 run. Only versions recorded as `empty` lose their empty document root; `no_root` and all bibliography rows are preserved. The original output, journal, benchmark and recovered BBL files are not changed. This does not rerun the parser. Run on a fresh Colab CPU runtime after this notebook and code are merged to `main`.

In [ ]:
from pathlib import Path

M2_DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1/m2"
M2_RUN_ID = "m2-full-01"
DERIVED_ID = "m2-full-no-empty-roots-01"  # Choose a new ID; existing destinations are refused.
LOCAL_ROOT = Path("/content/m2-empty-filter")
EXPECTED_SHARDS = 200
CODE_REF = "main"
REPO = "https://github.com/hcmus-arxiv-nhom1/CodeRepo.git"
RUN_FILTER = True  # Run the prepared full filter after checking Drive paths.

In [ ]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time

from google.colab import drive

drive.mount("/content/drive")
for label, value in (("run ID", M2_RUN_ID), ("derived ID", DERIVED_ID)):
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,127}", value):
        raise ValueError(f"Invalid {label}")
source = Path(M2_DRIVE_DIR) / M2_RUN_ID
destination = Path(M2_DRIVE_DIR) / "derived" / DERIVED_ID
if destination.exists():
    raise FileExistsError(f"Choose a new DERIVED_ID: {destination}")
if any(k == "src" or k.startswith("src.") for k in sys.modules):
    raise RuntimeError("Use a fresh Python kernel before selecting the revision")
repo = Path("/content/CodeRepo-filter")
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", REPO, str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "fetch", "origin", CODE_REF], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(repo)
sys.path.insert(0, str(repo))
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow", "tqdm"], check=True)
print("Filter revision:", revision)

## Restore and verify the original output

The Drive checkpoint is read only. All shard hashes and the completed run identity are verified. This can take time on a fresh runtime.

In [ ]:
from tqdm.auto import tqdm

from src.qa.qa_inputs import inspect_run, restore_shards

if RUN_FILTER:
    local_path = LOCAL_ROOT.resolve()
    for protected in (source.resolve(), destination.resolve()):
        if (
            local_path == protected
            or local_path in protected.parents
            or protected in local_path.parents
        ):
            raise ValueError("LOCAL_ROOT overlaps source or destination")
    if LOCAL_ROOT.exists():
        raise FileExistsError(f"Choose a clean LOCAL_ROOT: {LOCAL_ROOT}")
    identity, hashes = inspect_run(source, M2_RUN_ID, revision, EXPECTED_SHARDS)
    LOCAL_ROOT.mkdir(parents=True)
    with tqdm(total=len(hashes), desc="Restore original shards", unit="shard") as bar:
        restore_shards(
            source / "out",
            LOCAL_ROOT / "original",
            hashes,
            on_progress=lambda done, total: bar.update(done - bar.n),
        )
    journal_bytes = (source / "work/shards.jsonl").read_bytes()
    if hashlib.sha256(journal_bytes).hexdigest() != identity["journal_sha256"]:
        raise ValueError("Journal changed")
    journal = LOCAL_ROOT / "shards.jsonl"
    journal.write_bytes(journal_bytes)
    print("Original parser revision:", identity["parser_revision"])
else:
    print("Set RUN_FILTER = True to begin.")

## Verify, filter and check

The original is checked first. The derived output is checked with the journal's `empty` statuses explicitly requiring zero nodes. A failure prevents publication.

The journal retains all version statuses. The derived QA report counts versions represented in `hierarchy`; this count excludes the rootless `empty` versions and is not the total version coverage. `no_root` is intentionally unchanged.


In [ ]:
from src.qa.check_tables_bounded import check_run
from src.qa.filter_empty_roots import filter_output

if RUN_FILTER:
    started = time.perf_counter()
    with (
        tqdm(total=len(hashes), desc="Validate original partitions", unit="shard") as validation,
        tqdm(total=len(hashes), desc="Check original shards", unit="shard") as bar,
    ):
        original_report = check_run(
            LOCAL_ROOT / "original",
            journal,
            expected_hashes=hashes,
            on_validation=lambda done, total: validation.update(done - validation.n),
            on_progress=lambda done, total: bar.update(done - bar.n),
        )
    if not original_report["ok"]:
        raise ValueError(f"Original QA failed: {original_report}")
    with tqdm(total=len(hashes), desc="Filter shards", unit="shard") as bar:
        filter_report = filter_output(
            LOCAL_ROOT / "original",
            journal,
            LOCAL_ROOT / "filtered",
            on_progress=lambda done, total: bar.update(done - bar.n),
        )
    with tqdm(total=len(hashes), desc="Check derived shards", unit="shard") as bar:
        derived_report = check_run(
            LOCAL_ROOT / "filtered",
            journal,
            empty_without_root=True,
            on_progress=lambda done, total: bar.update(done - bar.n),
        )
    if not derived_report["ok"]:
        raise ValueError(f"Derived QA failed: {derived_report}")
    if (
        derived_report["rows"]["elements"]
        != original_report["rows"]["elements"] - filter_report["removed_roots"]
        or derived_report["rows"]["hierarchy"]
        != original_report["rows"]["hierarchy"] - filter_report["removed_roots"]
        or derived_report["rows"]["bibtex"] != original_report["rows"]["bibtex"]
    ):
        raise ValueError("Unexpected row count changes")
    print("Filter:", filter_report, "QA:", derived_report)
    print("Elapsed minutes:", round((time.perf_counter() - started) / 60, 1))

## Publish derived data to Drive

A new folder holds filtered Parquet, unchanged version journal, output hashes, identity and both QA reports. The original benchmark remains the original benchmark. Do not run two sessions with the same ID.

In [ ]:
from src.qa.qa_inputs import verify_checkpoint

if RUN_FILTER:
    current, current_hashes = inspect_run(source, M2_RUN_ID, revision, EXPECTED_SHARDS)
    if current != identity or current_hashes != hashes:
        raise ValueError("Source run changed during filtering")
    if destination.exists():
        raise FileExistsError(destination)
    output_hashes = {}
    for shard in sorted((LOCAL_ROOT / "filtered").glob("shard-*")):
        output_hashes[shard.name] = {}
        for path in sorted(shard.iterdir()):
            with path.open("rb") as f:
                output_hashes[shard.name][path.name] = hashlib.file_digest(f, "sha256").hexdigest()
    verify_checkpoint(LOCAL_ROOT / "filtered", journal, output_hashes)
    destination.parent.mkdir(parents=True, exist_ok=True)
    staging = destination.parent / ("." + DERIVED_ID + ".staging")
    if staging.exists():
        raise FileExistsError(f"Interrupted staging exists: {staging}")
    staging.mkdir()
    shutil.copytree(LOCAL_ROOT / "filtered", staging / "out")
    shutil.copy2(journal, staging / "shards.jsonl")
    if (
        hashlib.sha256((staging / "shards.jsonl").read_bytes()).hexdigest()
        != identity["journal_sha256"]
    ):
        raise ValueError("Published journal hash mismatch")
    for name, data in (
        ("identity.json", identity),
        ("shard-hashes.json", output_hashes),
        ("original_qa.json", original_report),
        ("derived_qa.json", derived_report),
        ("filter_report.json", filter_report),
    ):
        (staging / name).write_text(
            json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf8"
        )
    verify_checkpoint(staging / "out", staging / "shards.jsonl", output_hashes)
    if destination.exists():
        raise FileExistsError(destination)
    staging.rename(destination)
    print("Verified derived output:", destination)